# NYC Taxi Pipeline Demo

This notebook demonstrates the demo Spark pipeline against shared MinIO and Hive Metastore.

In [ ]:
from pyspark.sql import SparkSession
from pyspark.sql.functions import col

spark = (SparkSession.builder
    .appName('nyc-taxi-pipeline-demo')
    .config('spark.hadoop.fs.s3a.endpoint', 'http://minio:9000')
    .config('spark.hadoop.fs.s3a.access.key', os.environ.get('AWS_ACCESS_KEY_ID', ''))
    .config('spark.hadoop.fs.s3a.secret.key', os.environ.get('AWS_SECRET_ACCESS_KEY', ''))
    .config('spark.hadoop.fs.s3a.path.style.access', 'true')
    .config('spark.hadoop.fs.s3a.impl', 'org.apache.hadoop.fs.s3a.S3AFileSystem')
    .config('spark.hadoop.hive.metastore.uris', 'thrift://spark-infra-spark-35-metastore:9083')
    .config('spark.sql.warehouse.dir', 's3a://warehouse/spark-35')
    .enableHiveSupport()
    .getOrCreate())

spark

In [ ]:
df = spark.range(50000).withColumn('bucket', col('id') % 10)
df.write.mode('overwrite').parquet('s3a://spark-jobs/nyc-taxi-pipeline-demo/')
spark.read.parquet('s3a://spark-jobs/nyc-taxi-pipeline-demo/').groupBy('bucket').count().orderBy('bucket').show()

In [ ]:
spark.sql("CREATE DATABASE IF NOT EXISTS demo_shared LOCATION 's3a://warehouse/spark-35/demo_shared.db'")
spark.sql("DROP TABLE IF EXISTS demo_shared.taxi_buckets")
spark.sql("CREATE TABLE demo_shared.taxi_buckets USING PARQUET LOCATION 's3a://warehouse/spark-35/demo_shared.db/taxi_buckets' AS SELECT bucket, count(*) AS cnt FROM parquet.`s3a://spark-jobs/nyc-taxi-pipeline-demo/` GROUP BY bucket")
spark.sql("SELECT * FROM demo_shared.taxi_buckets ORDER BY bucket").show()